In [1]:
import pandas as pd

df = pd.concat([pd.read_csv(f'day_{i}.csv') for i in range(10) ])

df

,dac_family,qr,q,r,nx,valids,unique_dn,unique_valids,unique_nxd,unique_nxd_valids
0,conficker,21516,0,21516,7794,21516,499,499,367,367
1,modpack,1086,0,1086,0,1086,1,1,0,0
2,necurs,1315664,0,1315664,613894,1315664,14023,14023,13632,13632
3,pitou,5684,0,5684,2354,5684,20,20,17,17
4,suppobox,6,0,6,0,6,1,1,0,0
5,virut,66,0,66,0,66,1,1,0,0
6,NaN,33075694,0,33075694,994122,32130786,343312,298250,72332,28574
0,conficker,20385,0,20385,7698,20385,499,499,364,364
1,modpack,1704,0,1704,0,1704,2,2,0,0
2,necurs,1654929,0,1654929,804863,1654929,8053,8053,8021,8021


In [6]:

from sqlalchemy import create_engine


dbalchemy = create_engine(f"postgresql+psycopg2://postgres@localhost:5432/dns_mac")

def get(day):
    print(day)
    df = pd.read_sql(f"""
SELECT
	CASE WHEN cardinality(bigdn_m3.dac_families) > 0 AND bigdn_m3.dac_count_between > 0 THEN bigdn_m3.dac_families[1] ELSE NULL::text END
	AS dac_family,
	
	COUNT(*) AS QR,
	COUNT(*) filter (WHERE IS_R IS FALSE) AS Q,
	COUNT(*) filter (WHERE IS_R IS TRUE) AS R,
	COUNT(*) filter (WHERE RCODE=3) AS NX,
	COUNT(*) FILTER (WHERE bigdn_m3.regex_check) AS VALIDs,
	
	COUNT(DISTINCT DN_ID) AS UNIQUE_DN,
	COUNT(DISTINCT DN_ID) FILTER (WHERE bigdn_m3.regex_check) AS UNIQUE_VALIDS,
	COUNT(DISTINCT DN_ID) FILTER (WHERE RCODE=3) AS UNIQUE_NXD,
	COUNT(DISTINCT DN_ID) FILTER (WHERE RCODE=3 AND bigdn_m3.regex_check) AS UNIQUE_NXD_VALIDS
	
FROM
	MESSAGE3_it2016_{day} M2
	JOIN BIGDN_M3_2 BIGDN_M3 ON M2.DN_ID = BIGDN_M3.ID
GROUP BY
	DAC_FAMILY            
""", dbalchemy)
    
    values = df.to_numpy().tolist()
    for mw in [ 'conficker', 'modpack', 'necurs', 'pitou', 'suppobox', 'virut', None ]:
        if df['dac_family'].isin([mw]).sum() == 0:
            values.append([mw, 0,0,0,0,0,0,0,0,0])
    df = pd.DataFrame(values, columns=df.columns)
    df.to_csv(f'day__{day}.csv')
    df.insert(0, 'day', day)
    return df


pd.concat([ get(i) for i in range(10) ]).to_csv('days.csv')

0
1
2
3
4
5
6
7
8
9
